# Construction of VIIRS Historical Window Features

## Objective
This notebook constructs long-format VIIRS feature tables from the imputed VIIRS Zarr archive. The procedure is keyed by the existing wildfire label splits and generates historical predictors for each `(target_date, row, col)` sample.

## Historical Window Design
For each labeled spatial cell, the notebook extracts the previous 64 days of VIIRS history and partitions that interval into four causal windows of length 16 days. Summary statistics are then computed for the selected VIIRS arrays within each window.

## Output Structure
The exported feature files follow the long format:
- `target_date`
- `row`
- `col`
- `window_id`
- VIIRS-derived feature columns

These outputs are designed to align directly with the base feature tables and the split-specific wildfire label files.


## Environment and Imports

This section loads the feature-construction utilities and prepares the local project path so that the VIIRS window builder can be imported.


In [1]:
# Optional installs if needed
# %pip install -q numpy pandas zarr tqdm

from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from window_feature_builder_viirs import ViirsWindowFeatureConfig, main


## Configuration

This section defines the split-specific label files, the imputed VIIRS source archive, the output directory, and the historical aggregation settings used to generate the final feature tables.


In [2]:
# -------------------------
# Config
# -------------------------
cfg = ViirsWindowFeatureConfig(
    label_train_csv=Path('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_train.csv'),
    label_val_csv=Path('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_val.csv'),
    label_test_csv=Path('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_test.csv'),
    viirs_zarr_dir=Path('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs'),
    output_dir=Path('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features'),

    window_size=16,
    num_windows=4,
    history_days=64,

    arrays_to_use=('mean', 'min', 'max'),
    include_bands=None,
    exclude_bands=(),

    # Defaults: *_mean->mean, *_min->min, *_max->max
    aggregation_config={},
    min_valid_days_per_window=1,

    # Never drop datapoints, force neighbor/global fills if needed
    neighbor_radius=1,
    force_no_nan_output=True,

    n_jobs=6,

    # New output names so old files remain untouched
    out_all_name='FEATURES_viirs_all.csv',
    out_train_name='FEATURES_train_viirs.csv',
    out_val_name='FEATURES_val_viirs.csv',
    out_test_name='FEATURES_test_viirs.csv',
    out_summary_name='window_feature_viirs_summary.json',
    out_missing_stats_name='window_feature_viirs_missing_stats.csv',
    out_feature_list_name='feature_columns_viirs.csv',
)

cfg

ViirsWindowFeatureConfig(label_train_csv=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_train.csv'), label_val_csv=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_val.csv'), label_test_csv=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/LABELS_test.csv'), viirs_zarr_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/imputed_feature_zarr_viirs'), output_dir=PosixPath('/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features'), window_size=16, num_windows=4, history_days=64, include_bands=None, exclude_bands=(), arrays_to_use=('mean', 'min', 'max'), aggregation_config={}, default_agg='mean', ema_alpha=0.3, min_valid_days_per_window=1, neighbor_radius=1, force_no_nan_output=True, n_jobs=6, out_all_name='FEATURES_viirs_all.csv', out_train_name='FEATURES_train_viirs.csv', out_val_name='FEATURES_val_viirs.csv'

## Feature Construction Run

The configured VIIRS window-feature pipeline is executed here. The returned paths identify the generated split files, combined output file, and summary diagnostics.


In [3]:
# -------------------------
# Run pipeline
# -------------------------
out_paths = main(cfg)
print(json.dumps(out_paths, indent=2))

[2026-04-30 21:48:15] INFO | Processing VIIRS features=18 with workers=6
[2026-04-30 21:49:14] INFO | Done i1_mean | nan_after=0
[2026-04-30 21:49:15] INFO | Done evi_min | nan_after=0
[2026-04-30 21:49:15] INFO | Done evi_max | nan_after=0
[2026-04-30 21:49:15] INFO | Done evi_mean | nan_after=0
[2026-04-30 21:49:18] INFO | Done i1_min | nan_after=0
[2026-04-30 21:49:18] INFO | Done i1_max | nan_after=0
[2026-04-30 21:50:17] INFO | Done i2_mean | nan_after=0
[2026-04-30 21:50:19] INFO | Done i3_mean | nan_after=0
[2026-04-30 21:50:24] INFO | Done i2_min | nan_after=0
[2026-04-30 21:50:24] INFO | Done i2_max | nan_after=0
[2026-04-30 21:50:27] INFO | Done i3_min | nan_after=0
[2026-04-30 21:50:28] INFO | Done i3_max | nan_after=0
[2026-04-30 21:51:22] INFO | Done m11_mean | nan_after=0
[2026-04-30 21:51:27] INFO | Done m11_min | nan_after=0
[2026-04-30 21:51:28] INFO | Done ndvi_mean | nan_after=0
[2026-04-30 21:51:30] INFO | Done ndvi_min | nan_after=0
[2026-04-30 21:51:30] INFO | Don

{
  "all": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_viirs_all.csv",
  "train": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_train_viirs.csv",
  "val": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_val_viirs.csv",
  "test": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/FEATURES_test_viirs.csv",
  "missing_stats": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/window_feature_viirs_missing_stats.csv",
  "feature_list": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/feature_columns_viirs.csv",
  "summary": "/Users/copperhead07/Computing/DataCollection/RealWork/FEATURE/window_features/window_feature_viirs_summary.json"
}


## Output Inspection

This section provides a compact inspection of the generated split files and allows a direct check of the long-format VIIRS feature layout.


In [4]:
# -------------------------
# Quick preview
# -------------------------
import pandas as pd

train_df = pd.read_csv(out_paths['train'])
val_df = pd.read_csv(out_paths['val'])
test_df = pd.read_csv(out_paths['test'])

print('train rows:', len(train_df))
print('val rows:', len(val_df))
print('test rows:', len(test_df))
display(train_df.head(8))
display(val_df.head(8))
display(test_df.head(8))

train rows: 190120
val rows: 41160
test rows: 7560


,target_date,row,col,window_id,evi_mean,evi_min,evi_max,i1_mean,i1_min,i1_max,...,i2_max,i3_mean,i3_min,i3_max,m11_mean,m11_min,m11_max,ndvi_mean,ndvi_min,ndvi_max
0,2021-04-09,108,60,1,-5.776077,-4060.000000,598.930600,0.127762,0.0000,0.6826,...,0.716000,0.178741,0.0,0.610800,0.154144,0.0,0.521000,0.219063,0.000000,0.655594
1,2021-04-09,108,60,2,0.155674,-76.186440,45.495050,0.259791,-0.0100,1.0438,...,1.084500,0.157256,0.0,0.659300,0.204548,0.0,0.535400,0.143980,-0.007226,0.660384
2,2021-04-09,108,60,3,-0.103728,-316.071440,182.941180,0.085793,-0.0015,0.3391,...,0.726912,0.158945,0.0,0.496302,0.118624,0.0,0.365225,0.259800,0.000000,0.628111
3,2021-04-09,108,60,4,0.356151,-28.817734,76.382980,0.225511,0.0000,0.9599,...,0.940600,0.168236,0.0,0.526100,0.142121,0.0,0.535962,0.158065,-0.012937,0.574803
4,2021-02-25,68,49,1,-1.688333,-1338.750000,15.034602,0.155467,-0.0100,0.8538,...,0.870400,0.165963,0.0,0.632400,0.119019,0.0,0.535600,0.184391,-0.363918,0.820380
5,2021-02-25,68,49,2,-0.007522,-48.750000,16.352942,0.271026,-0.0100,0.9809,...,0.966000,0.157523,0.0,0.480000,0.157189,0.0,0.522000,0.050533,-0.440407,0.733189
6,2021-02-25,68,49,3,0.031909,-0.116825,14.282116,0.159809,0.0000,0.9554,...,0.943300,0.115383,0.0,0.657614,0.102542,0.0,0.678515,0.059378,-0.449082,0.554851
7,2021-02-25,68,49,4,0.083496,-1.827889,3.647687,0.209647,0.0000,1.0427,...,1.015500,0.174821,0.0,0.636600,0.172583,0.0,0.546400,0.019396,-0.413333,0.535248


,target_date,row,col,window_id,evi_mean,evi_min,evi_max,i1_mean,i1_min,i1_max,...,i2_max,i3_mean,i3_min,i3_max,m11_mean,m11_min,m11_max,ndvi_mean,ndvi_min,ndvi_max
0,2019-07-04,135,54,1,2.609654,-309.000000,1195.000000,0.080932,-0.0071,0.501500,...,0.561000,0.177846,0.0,0.571600,0.157227,0.0000,0.566500,0.266290,-0.068638,0.710499
1,2019-07-04,135,54,2,-2.099825,-2822.500000,166.857150,0.127120,0.0000,1.097000,...,1.110300,0.211956,0.0,0.710300,0.160311,0.0000,0.539500,0.269924,-0.385366,0.602066
2,2019-07-04,135,54,3,0.172559,-117.121216,460.833340,0.263515,-0.0100,0.869300,...,0.877800,0.210377,0.0,0.769500,0.191385,0.0000,0.601900,0.210859,-0.406522,0.873786
3,2019-07-04,135,54,4,-0.759808,-289.736850,200.434780,0.178945,0.0000,0.780300,...,0.883500,0.221518,0.0,0.531000,0.157414,0.0584,0.450300,0.288195,-0.392816,0.638591
4,2018-08-07,63,106,1,0.662224,-23.125000,87.380950,0.114575,0.0000,0.793452,...,0.815685,0.247605,0.0,0.633435,0.185591,0.0000,0.465768,0.122476,-0.019847,0.212136
5,2018-08-07,63,106,2,0.244810,-9.841897,54.534885,0.217813,0.0000,1.101600,...,1.120800,0.193179,0.0,0.660500,0.203021,0.0000,0.525800,0.075205,-0.012920,0.343257
6,2018-08-07,63,106,3,0.440586,-74.054054,119.347824,0.143387,0.0000,0.240000,...,0.290900,0.227564,0.0,0.632033,0.176566,0.0000,0.484650,0.104774,0.000000,0.351802
7,2018-08-07,63,106,4,0.851913,-199.500000,361.875000,0.142359,0.0000,0.827200,...,0.865500,0.226779,0.0,0.846200,0.179300,0.0000,0.626200,0.109854,0.000000,0.321895


,target_date,row,col,window_id,evi_mean,evi_min,evi_max,i1_mean,i1_min,i1_max,...,i2_max,i3_mean,i3_min,i3_max,m11_mean,m11_min,m11_max,ndvi_mean,ndvi_min,ndvi_max
0,2023-07-30,113,65,1,2.414277,-193.71213,880.00000,0.102449,0.0000,0.330300,...,0.626800,0.246256,0.1379,0.430100,0.161009,0.0000,0.347900,0.445917,0.000000,0.936502
1,2023-07-30,113,65,2,-6.357824,-2139.00000,1104.16660,0.094382,0.0224,0.509500,...,0.617600,0.252975,0.0805,0.597400,0.157930,0.0346,0.353800,0.472104,0.042099,0.880716
2,2023-07-30,113,65,3,0.255595,-1252.00000,701.00000,0.134988,-0.0100,0.920468,...,0.828000,0.209832,0.0000,0.535800,0.143550,0.0000,0.473700,0.463943,-0.011168,0.951327
3,2023-07-30,113,65,4,1.627579,-146.66667,236.73914,0.263585,-0.0100,0.981300,...,0.969300,0.229849,0.0000,0.628300,0.195887,0.0000,0.528600,0.346718,-0.008703,0.941392
4,2023-08-23,108,34,1,0.791325,-120.37500,351.25000,0.164419,0.0000,0.952198,...,0.966931,0.156840,0.0000,0.658500,0.125678,0.0000,0.490700,0.445427,-0.026935,0.953719
5,2023-08-23,108,34,2,0.894395,-371.60000,139.21875,0.088967,-0.0010,0.802700,...,0.842600,0.165635,0.0264,0.687300,0.114081,0.0219,0.508600,0.503797,0.020912,0.856365
6,2023-08-23,108,34,3,0.457393,-592.85710,500.38460,0.066311,0.0000,0.877615,...,0.345000,0.181683,0.0526,0.342295,0.112488,0.0000,0.236531,0.609144,0.000000,0.856639
7,2023-08-23,108,34,4,-5.515661,-4970.00000,451.66666,0.123292,-0.0100,1.005800,...,0.990200,0.170261,0.0000,0.439500,0.122171,0.0000,0.414700,0.524990,-0.009422,0.902212
